In [ ]:
import torch
import pandas as pd
import os
from tqdm.auto import tqdm
from google.colab import drive
drive.mount("/content/drive")
TOP_K = 3
neuron_lists = {}

import torch.nn as nn
import torch.nn.functional as F

class SAE(nn.Module):
    def __init__(self, d_in, d_hidden, l1_coeff=3e-4):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_hidden, bias=False)
        self.decoder = nn.Linear(d_hidden, d_in, bias=False)
        self.l1_coeff = l1_coeff

        # Tied weights option (recommended for interpretability)
        self.decoder.weight = nn.Parameter(self.encoder.weight.T)

    def forward(self, x):
        # Sparse latent features
        z = F.relu(self.encoder(x))
        # Reconstruction
        x_hat = self.decoder(z)
        return x_hat, z

    def loss(self, x, x_hat, z):
        mse = ((x - x_hat)**2).mean()
        l1 = self.l1_coeff * z.abs().sum(dim=-1).mean()
        return mse + l1


MessageError: Error: credential propagation was unsuccessful

In [ ]:
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from contextlib import contextmanager
from google.colab import drive
drive.mount("/content/drive")

# ----------------------------------
# Load intervention specification
# ----------------------------------
spec = torch.load("/content/drive/MyDrive/SAE_Works_Pythia-2.8/layer 24/reward_hacking_intervention.pt")

MODEL_NAME = spec["model"]
LAYER_IDX  = spec["layer"]
ALPHA      = spec["alpha"]

# ----------------------------------
# Load model + tokenizer
# ----------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

# ----------------------------------
# Load SAEs
# ----------------------------------
saes = []
for sae_file in spec["sae_files"]:
    sae = torch.load(f"/content/drive/MyDrive/SAE_Works_Pythia-2.8B/layer 24/sae_mixed{sae_id+1}.pt", map_location="cuda", weights_only=False)
    sae.eval().cuda()
    saes.append(sae)

# ----------------------------------
# Load neuron rankings
# ----------------------------------
neuron_lists = {}

for i, csv_path in enumerate(spec["neuron_csvs"]):
    df = pd.read_csv(csv_path)
    neuron_lists[i] = df["neuron_id"].tolist()


In [ ]:
class MatryoshkaSAEIntervention:
    def __init__(self, saes, neuron_lists, alpha):
        self.saes = saes
        self.neuron_lists = neuron_lists
        self.alpha = alpha

    def __call__(self, acts, hook):
        # acts: [batch, seq, d_model]
        B, S, D = acts.shape
        x = acts.reshape(-1, D).float().cuda()

        with torch.no_grad():
            for sae_id, sae in enumerate(self.saes):
                _, z = sae(x)

                for idx in self.neuron_lists[sae_id]:
                    z[:, idx] *= self.alpha

                x = sae.decoder(z)

        return x.reshape(B, S, D)


In [ ]:
@contextmanager
def apply_intervention(model, intervention):
    hook_handle = model.gpt_neox.layers[LAYER_IDX].mlp.register_forward_hook(intervention)
    try:
        yield
    finally:
        hook_handle.remove()


In [ ]:
def run_rewardbench(model, dataset):
    results = []
    for sample in dataset:
        inputs = tokenizer(sample["prompt"], return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=128)
        completion = tokenizer.decode(out[0], skip_special_tokens=True)
        results.append(completion)
    return results

baseline_outputs = run_rewardbench(model, rewardbench_dataset)


In [ ]:
intervention = MatryoshkaSAEIntervention(
    saes=saes,
    neuron_lists=neuron_lists,
    alpha=ALPHA
)

with apply_intervention(model, intervention):
    intervened_outputs = run_rewardbench(model, rewardbench_dataset)
